<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/ACSIncome_Stability_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import KBinsDiscretizer, LabelEncoder

from pgmpy.estimators import PC, HillClimbSearch, BDeu
from pgmpy.models import DiscreteBayesianNetwork

from folktables import ACSDataSource, ACSIncome


# -----------------------------
# 1) Load ACSIncome
# -----------------------------

def load_acs_income(
    survey_year: str = "2018",
    horizon: str = "1-Year",
    state: Optional[str] = None,
) -> pd.DataFrame:
    """
    Load ACSIncome. If state is None, loads all states available via the data source.
    For speed, you may set state="CA" or similar.

    Returns a DataFrame with Folktables features + target column 'income' (0/1).
    """
    data_source = ACSDataSource(survey_year=survey_year, horizon=horizon, survey="person")
    acs_data = data_source.get_data(states=[state] if state else None, download=True)

    # Folktables returns numpy arrays; we rebuild a DataFrame
    X, y, _ = ACSIncome.df_to_numpy(acs_data)

    feature_names = ACSIncome.features
    df = pd.DataFrame(X, columns=feature_names)
    df["income"] = y.astype(int)
    return df


# -----------------------------
# 2) Preprocess to discrete ints
# -----------------------------

@dataclass
class PreprocessConfig:
    n_bins: int = 5
    bin_strategy: str = "quantile"  # 'uniform' or 'quantile'
    max_rows: Optional[int] = None  # optionally subsample for speed


def preprocess_discrete_for_pgmpy(df: pd.DataFrame, target_col: str, cfg: PreprocessConfig) -> Tuple[pd.DataFrame, List[str]]:
    """
    Convert all columns to discrete integer-coded columns suitable for pgmpy.
    - Numeric columns -> KBinsDiscretizer (ordinal bins)
    - Non-numeric -> LabelEncoder

    Returns (discrete_df, feature_cols)
    """
    out = df.copy()

    # Optional row subsample for speed (keeps randomness deterministic)
    if cfg.max_rows is not None and len(out) > cfg.max_rows:
        out = out.sample(n=cfg.max_rows, random_state=42).reset_index(drop=True)

    # Fill missing values (Folktables generally has none, but be safe)
    for col in out.columns:
        if out[col].isna().any():
            mode_val = out[col].mode(dropna=True)
            fill_val = mode_val.iloc[0] if not mode_val.empty else 0
            out[col] = out[col].fillna(fill_val)

    # Identify numeric columns excluding target
    numeric_cols = [c for c in out.columns if c != target_col and pd.api.types.is_numeric_dtype(out[c])]
    cat_cols = [c for c in out.columns if c not in numeric_cols]

    # Discretize numeric columns
    if numeric_cols:
        binner = KBinsDiscretizer(
            n_bins=cfg.n_bins,
            encode="ordinal",
            strategy=cfg.bin_strategy,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            out[numeric_cols] = binner.fit_transform(out[numeric_cols])
        out[numeric_cols] = out[numeric_cols].astype(int)

    # Label encode categorical columns (including target if it ends up non-numeric)
    for c in cat_cols:
        le = LabelEncoder()
        out[c] = le.fit_transform(out[c].astype(str))

    feature_cols = [c for c in out.columns if c != target_col]
    return out, feature_cols


# -----------------------------
# 3) Graph learning helpers
# -----------------------------

def pc_stable_markov_blanket(data: pd.DataFrame, target_col: str, alpha: float, max_cond_vars: int = 2) -> List[str]:
    """
    Learn a structure with PC-Stable and return MB(target_col).
    """
    est = PC(data)
    model = est.estimate(
        variant="stable",
        ci_test="chi_square",
        significance_level=alpha,
        max_cond_vars=max_cond_vars,
        show_progress=False,
    )

    # Ensure we have a DAG-like edge list
    try:
        dag = model.to_dag()
        edges = list(dag.edges())
    except Exception:
        edges = list(model.edges())

    bn = DiscreteBayesianNetwork(edges)
    mb = bn.get_markov_blanket(target_col)
    return sorted(set(mb))


def hc_bdeu_markov_blanket(
    data: pd.DataFrame,
    target_col: str,
    ess: int = 10,
    max_indegree: int = 2,
) -> List[str]:
    """
    Learn a structure with Hill-Climbing + BDeu and return MB(target_col).
    """
    scoring = BDeu(data, equivalent_sample_size=ess)
    est = HillClimbSearch(data)
    model = est.estimate(scoring_method=scoring, max_indegree=max_indegree, show_progress=False)

    edges = list(model.edges())
    bn = DiscreteBayesianNetwork(edges)
    mb = bn.get_markov_blanket(target_col)
    return sorted(set(mb))


# -----------------------------
# 4) Bootstrap stability
# -----------------------------

def stability_analysis_pc_hc(
    data: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    pc_alpha: float,
    hc_max_indegree: int,
    n_bootstraps: int = 50,
    sample_frac: float = 0.30,
    random_state: int = 42,
    # PC params
    pc_max_cond_vars: int = 2,
    # HC params
    hc_ess: int = 10,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs bootstrap stability and returns:
      - summary_df: feature inclusion frequency in MB(target) for PC and HC
      - raw_df: per-bootstrap results
    """
    rng = np.random.default_rng(random_state)
    n = len(data)

    rows = []
    for b in tqdm(range(1, n_bootstraps + 1), desc="Bootstraps"):
        idx = rng.choice(n, size=int(sample_frac * n), replace=True)
        sample = data.iloc[idx].reset_index(drop=True)

        # PC-Stable MB
        try:
            mb_pc = pc_stable_markov_blanket(sample, target_col, alpha=pc_alpha, max_cond_vars=pc_max_cond_vars)
        except Exception as e:
            mb_pc = []
            print(f"[Warning] PC failed at bootstrap {b}: {e}")

        for f in feature_cols:
            rows.append({"bootstrap": b, "algorithm": "PC-Stable", "feature": f, "selected": int(f in mb_pc)})

        # HC-BDeu MB
        try:
            mb_hc = hc_bdeu_markov_blanket(sample, target_col, ess=hc_ess, max_indegree=hc_max_indegree)
        except Exception as e:
            mb_hc = []
            print(f"[Warning] HC failed at bootstrap {b}: {e}")

        for f in feature_cols:
            rows.append({"bootstrap": b, "algorithm": "HC-BDeu", "feature": f, "selected": int(f in mb_hc)})

    raw_df = pd.DataFrame(rows)

    summary_df = (
        raw_df.groupby(["algorithm", "feature"], as_index=False)
        .agg(selection_frequency=("selected", "mean"), variance=("selected", "var"), n=("selected", "count"))
        .sort_values(["algorithm", "selection_frequency"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return summary_df, raw_df


# -----------------------------
# 5) Run
# -----------------------------

if __name__ == "__main__":
    # (A) Load data. For speed, you may set state="CA" or another state abbreviation.
    df = load_acs_income(survey_year="2018", horizon="1-Year", state='CA')
    target_col = "income"

    # (B) Preprocess to discrete
    cfg = PreprocessConfig(n_bins=5, bin_strategy="quantile", max_rows=30000)  # set max_rows=50000 to speed up
    disc_df, feature_cols = preprocess_discrete_for_pgmpy(df, target_col, cfg)

    print("ACSIncome loaded:", disc_df.shape)
    print("Features:", feature_cols)


    import pandas as pd

    # Define the parameter ranges for the sweep
    pc_alphas = [0.01, 0.05, 0.10]
    hc_indegrees = [2, 3, 4]

    # Storage for aggregated results
    all_summaries = []
    all_raw_data = []

    print("Starting Stability Sweep...")

    for alpha in pc_alphas:
        for indegree in hc_indegrees:
            print(f"\n>>> Testing: PC Alpha={alpha}, HC Max Indegree={indegree}")

            # (C) Stability analysis
            summary, raw = stability_analysis_pc_hc(
              data=disc_df,
              target_col=target_col,
              feature_cols=feature_cols,
              n_bootstraps=50,
              sample_frac=0.30,
              random_state=42,
              pc_alpha=alpha,           # Iterative parameter
              pc_max_cond_vars=2,
              hc_ess=10,
              hc_max_indegree=indegree, # Iterative parameter
            )

            # Add metadata for tracking
            summary['pc_alpha'] = alpha
            summary['hc_max_indegree'] = indegree
            raw['pc_alpha'] = alpha
            raw['hc_max_indegree'] = indegree

            all_summaries.append(summary)
            all_raw_data.append(raw)

            # (D) Inspect local results for this specific iteration
            print(f"--- Stability summary for Alpha={alpha}, Indegree={indegree} ---")
            for algo in ["PC-Stable", "HC-BDeu"]:
                print(f"\n[{algo}]")
                print(summary[summary["algorithm"] == algo].head(10).to_string(index=False))

    # Concatenate all results into single DataFrames
    final_summary = pd.concat(all_summaries, ignore_index=True)
    final_raw = pd.concat(all_raw_data, ignore_index=True)

    # Save the consolidated results
    final_summary.to_csv("acs_income_mb_stability_sweep_summary.csv", index=False)
    final_raw.to_csv("acs_income_mb_stability_sweep_raw.csv", index=False)

    print("\n" + "="*40)
    print("Sweep Complete. Consolidated files saved:")
    print(" - acs_income_mb_stability_sweep_summary.csv")
    print(" - acs_income_mb_stability_sweep_raw.csv")


ACSIncome loaded: (30000, 11)
Features: ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'RAC1P']
Starting Stability Sweep...

>>> Testing: PC Alpha=0.01, HC Max Indegree=2


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/CITests.py:425: FutureWarning: `power_divergence` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.PowerDivergence`
        instead.
  return power_divergence(X=X, Y=Y, Z=Z, data=data, boolean=boolean, lambda_="pearson", **kwargs)
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/BaseConstraintEstimator.py:219: FutureWarning: `chi_square` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.ChiSquare` instead.
  if ci_test(
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/CITests.py:425: FutureWarning: `power_divergence` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.PowerDivergence`
        instead.
  return power_divergence(X=X, Y=Y, Z=Z, data=data, boolean=boolean, lambda_="pearson", **kwargs)
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/BaseConstraintEstimator.py:219: FutureWarning: 

[Warning] PC failed at bootstrap 13: Input is not a valid edge list


Streaming output truncated to the last 5000 lines.
  return power_divergence(X=X, Y=Y, Z=Z, data=data, boolean=boolean, lambda_="pearson", **kwargs)
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/BaseConstraintEstimator.py:219: FutureWarning: `chi_square` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.ChiSquare` instead.
  if ci_test(
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/CITests.py:425: FutureWarning: `power_divergence` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.PowerDivergence`
        instead.
  return power_divergence(X=X, Y=Y, Z=Z, data=data, boolean=boolean, lambda_="pearson", **kwargs)
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/BaseConstraintEstimator.py:219: FutureWarning: `chi_square` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.ChiSquare` instead.
  if ci_test(
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/CITests.py:425: FutureWarning: `power

[Warning] PC failed at bootstrap 25: Input is not a valid edge list


Bootstraps:  50%|█████     | 25/50 [07:49<07:44, 18.56s/it]/tmp/ipykernel_2361/2165129511.py:109: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/BaseConstraintEstimator.py:219: FutureWarning: `chi_square` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.ChiSquare` instead.
  if ci_test(
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/CITests.py:425: FutureWarning: `power_divergence` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.PowerDivergence`
        instead.
  return power_divergence(X=X, Y=Y, Z=Z, data=data, boolean=boolean, lambda_="pearson", **kwargs)
/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/BaseConstraintEstimator.py:219: FutureWarning: `chi_square` is deprecated and will be removed in v1.3.0. Please use `pgmpy.ci_tests.ChiSquare` instead.
  if ci_test(
/usr/local/lib/pyth

KeyboardInterrupt: 